<a href="https://colab.research.google.com/github/pranatixsharma/Masculine_defaults_Indian_youtube/blob/main/get_transcripts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Import models

In [ ]:
!pip install whisperx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

AUDIO_DIR  = "/content/drive/MyDrive/audio/politics_news"
OUTPUT_DIR = "/content/drive/MyDrive/transcripts_roman/politics_transcripts"
DEVICE     = "cuda"
BATCH_SIZE = 16


In [ ]:
#corruption check
import json
import glob
import os

def remove_error_jsons(folder):
    removed = []

    for path in glob.glob(os.path.join(folder, "*.json")):
        try:
            with open(path, encoding="utf-8") as f:
                data = json.load(f)

            # Remove any JSON file that contains an "error" field
            if isinstance(data, dict) and "error" in data:
                os.remove(path)
                removed.append(os.path.basename(path))

        except json.JSONDecodeError:
            # Also remove malformed JSON files
            os.remove(path)
            removed.append(os.path.basename(path))

    print(f"Removed {len(removed)} files.")
    for f in removed:
        print(f"  {f}")

# Use your folder variable
remove_error_jsons("/content/drive/MyDrive/transcripts_roman")



Removed 0 files.


In [ ]:
#load models & GPU warmup
import whisperx
import numpy as np
import torch

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading Whisper model...")
whisper_model = whisperx.load_model("large-v2", DEVICE, compute_type="float16")

print("Loading alignment model...")
align_model, metadata = whisperx.load_align_model(language_code="hi", device=DEVICE)

# GPU warmup — prevents CUDA errors on first real file
print("Warming up GPU...")
dummy = np.zeros(16000, dtype=np.float32)
whisper_model.transcribe(dummy, batch_size=BATCH_SIZE)
print("Ready")


Loading Whisper model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocabulary.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Warming up GPU...


/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(


2026-08-12 16:38:18 - whisperx.vads.pyannote - WARNING - No active speech found in audio
2026-08-12 16:38:18 - whisperx.asr - WARNING - Audio is shorter than 30s, language detection may be inaccurate
2026-08-12 16:38:19 - whisperx.asr - INFO - Detected language: en (0.41) in first 30s of audio
Ready


In [ ]:
#transcription helpers
import time

def transcribe_with_retry(path, retries=3, wait=10):
    for attempt in range(retries):
        try:
            audio = whisperx.load_audio(path)
            raw = whisper_model.transcribe(audio, batch_size=BATCH_SIZE)
            aligned = whisperx.align(
                raw["segments"], align_model, metadata, audio, DEVICE
            )
            return aligned, raw.get("language", "")
        except RuntimeError as e:
            if "cuda" in str(e).lower() and attempt < retries - 1:
                print(f"  CUDA error (attempt {attempt+1}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

def build_output(filename, language, aligned):
    return {
        "filename": filename,
        "language": language,
        "segments": [
            {
                "start": s["start"],
                "end":   s["end"],
                "text":  s["text"],
                "words": [
                    {
                        "word":  w["word"],
                        "start": w.get("start"),
                        "end":   w.get("end"),
                        "score": w.get("score")
                    }
                    for w in s.get("words", [])
                ]
            }
            for s in aligned["segments"]
        ]
    }


In [ ]:
#main batch loop


from tqdm.auto import tqdm

EXTENSIONS = ("*.wav", "*.mp3", "*.m4a", "*.flac")

audio_files = sorted(
    f for ext in EXTENSIONS
    for f in glob.glob(os.path.join(AUDIO_DIR, ext))
)

already_done = {
    os.path.basename(p).replace(".json", "")
    for p in glob.glob(OUTPUT_DIR + "/*.json")
}

pending = [
    p for p in audio_files
    if os.path.basename(p).rsplit(".", 1)[0] not in already_done
]

print(f"Total files   : {len(audio_files)}")
print(f"Already done  : {len(already_done)}")
print(f"To process    : {len(pending)}")

for path in tqdm(pending):
    fname    = os.path.basename(path)
    out_path = os.path.join(OUTPUT_DIR, fname.rsplit(".", 1)[0] + ".json")

    try:
        aligned, language = transcribe_with_retry(path)
        result = build_output(fname, language, aligned)
    except Exception as e:
        result = {"filename": fname, "error": str(e)}

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

print("Session complete.")


Total files   : 1414
Already done  : 1348
To process    : 150


  0%|          | 0/150 [00:00<?, ?it/s]

2026-08-12 16:41:44 - whisperx.asr - INFO - Detected language: hi (0.97) in first 30s of audio
2026-08-12 16:42:53 - whisperx.asr - INFO - Detected language: hi (0.95) in first 30s of audio
2026-08-12 16:44:00 - whisperx.asr - INFO - Detected language: hi (0.91) in first 30s of audio
2026-08-12 16:45:12 - whisperx.asr - INFO - Detected language: hi (0.95) in first 30s of audio
2026-08-12 16:46:25 - whisperx.asr - INFO - Detected language: hi (0.95) in first 30s of audio
2026-08-12 16:47:35 - whisperx.asr - INFO - Detected language: en (0.88) in first 30s of audio
2026-08-12 16:48:21 - whisperx.asr - INFO - Detected language: hi (0.99) in first 30s of audio
2026-08-12 16:49:28 - whisperx.asr - INFO - Detected language: hi (0.84) in first 30s of audio
2026-08-12 16:50:39 - whisperx.asr - INFO - Detected language: hi (0.94) in first 30s of audio
2026-08-12 16:51:44 - whisperx.asr - INFO - Detected language: hi (0.98) in first 30s of audio
2026-08-12 16:52:54 - whisperx.asr - INFO - Detect

In [ ]:
#progress check
done   = glob.glob(OUTPUT_DIR + "/*.json")
errors = []
ok     = []

for p in done:
    with open(p, encoding="utf-8") as f:
        d = json.load(f)
    (errors if "error" in d else ok).append(os.path.basename(p))

print(f"Successfully transcribed : {len(ok)}")
print(f"Errors                   : {len(errors)}")
print(f"Remaining                : {len(audio_files) - len(done)}")

if errors:
    print("\nError files:")
    for f in errors: print(f"  {f}")


Successfully transcribed : 1498
Errors                   : 0
Remaining                : -84
